In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import torch
assert torch.cuda.is_available(), 'no GPU: switch the runtime to a GPU and rerun'
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
assert torch.cuda.get_device_capability(0)[0] >= 8, 'no native bfloat16: use an L4 or A100 runtime'

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/temporal-ood-ddi'
os.makedirs(f'{D}/runs/preds', exist_ok=True)
os.makedirs(f'{D}/tac2018', exist_ok=True)
print(D)

In [ ]:
!mkdir -p /content/w && cp {D}/rev3_bundle.tgz /content/rev3_bundle.tgz && tar xzf /content/rev3_bundle.tgz -C /content/w
!pip -q install bitsandbytes accelerate
%cd /content/w

In [ ]:
JOBS = [
 "r3-olmo-tac-calibration"
]

In [ ]:
import hashlib, json, sys
sys.path.insert(0, '/content/w')
from ddiood import rev2_adjudicator as ra, rev3_jobs as rj, rev3_tac as rt
assert ra.prompt_sha256() == ra.EXPECTED_PROMPT_SHA256, 'frozen R2b prompt hash differs'
assert ra.OLMO_REVISION == rj.PLAN_OLMO_REVISION, 'model revision differs from the plan'
dirs = rt.fetch(f'{D}/tac2018')
items, sections = rt.build_all(dirs)
assert rt.items_sha256(items) == rj.EXPECTED_TAC_ITEMS_SHA256, 'TAC item list differs from the frozen list'
print(len(items), 'items', rt.items_sha256(items))

In [ ]:
%cd /content/w
!bash -c "set -o pipefail; python -m ddiood.rev3_jobs run --tac-dir {D}/tac2018 --out {D}/runs --only {' '.join(JOBS)} 2>&1 | tee -a {D}/runs/r3-R2c_tac_calibration.log | (grep --line-buffered -E '^(start|done|skip|FAILED)|Error|Traceback'; true)"

In [ ]:
import os
done = sorted(j for j in JOBS if os.path.exists(f'{D}/runs/preds/{j}.json'))
failed = sorted(j for j in JOBS if os.path.exists(f'{D}/runs/preds/{j}.failed.txt'))
print(f'{len(done)}/{len(JOBS)} jobs finished')
print('missing:', [j for j in JOBS if j not in done])
print('failed:', failed)
for j in failed:
    print(j, open(f'{D}/runs/preds/{j}.failed.txt').read()[-1500:])
assert not failed, failed
assert len(done) == len(JOBS), [j for j in JOBS if j not in done]

In [ ]:
from google.colab import drive
drive.flush_and_unmount()